# Topic Modeling (LDA)

In [1]:
import re
import pandas as pd
import nltk
import gensim
from gensim import corpora
from gensim.models import CoherenceModel
 
nltk.download("stopwords", quiet=True)
from nltk.corpus import stopwords

In [2]:
df = pd.read_csv("../data/processed/cleaned_restaurant_reviews.csv")
stop_words = set(stopwords.words("english"))

_WORD_RE = re.compile(r"\b[a-z]+\b")
 
def tokenize(text):
    text = str(text).lower()
    words = _WORD_RE.findall(text)
    return [w for w in words if w not in stop_words and len(w) > 2]
 
texts = df["review_clean"].apply(tokenize).tolist()
texts[:2]

[['good', 'ambience', 'friendly', 'staff', 'great', 'food'],
 ['food',
  'vibe',
  'must',
  'try',
  'coconut',
  'pudding',
  'ultimate',
  'liked',
  'biriyani']]

### Build dictionary and corpus

In [3]:
dictionary = corpora.Dictionary(texts)
dictionary.filter_extremes(no_below=3, no_above=0.5)
corpus = [dictionary.doc2bow(t) for t in texts]
print("Vocabulary size:", len(dictionary))

Vocabulary size: 396


In [4]:
coherence_scores = {}
for k in [3, 4, 5, 6, 7, 8]:
    model_k = gensim.models.LdaModel(
        corpus=corpus, id2word=dictionary, num_topics=k,
        random_state=42, passes=15, alpha="auto",
    )
    cm = CoherenceModel(model=model_k, texts=texts, dictionary=dictionary, coherence="c_v")
    coherence_scores[k] = cm.get_coherence()
    print(f"num_topics={k}  coherence={coherence_scores[k]:.4f}")
 
best_k = max(coherence_scores, key=coherence_scores.get)
print(f"\nBest num_topics by coherence: {best_k}")

num_topics=3  coherence=0.2954
num_topics=4  coherence=0.2773
num_topics=5  coherence=0.3064
num_topics=6  coherence=0.3143
num_topics=7  coherence=0.3375
num_topics=8  coherence=0.3468

Best num_topics by coherence: 8


### Train the LDA model

In [5]:
num_topics = best_k  # or set manually, e.g. num_topics = 5
lda_model = gensim.models.LdaModel(
    corpus=corpus, id2word=dictionary, num_topics=num_topics,
    random_state=42, passes=15, alpha="auto"
)
for idx, topic in lda_model.print_topics(num_words=8):
    print(f"Topic {idx}: {topic}")

Topic 0: 0.091*"food" + 0.052*"service" + 0.042*"experience" + 0.042*"quality" + 0.040*"poor" + 0.034*"good" + 0.031*"staff" + 0.031*"disappointing"
Topic 1: 0.030*"loved" + 0.030*"one" + 0.028*"dishes" + 0.027*"place" + 0.025*"experience" + 0.022*"ordered" + 0.022*"quantity" + 0.020*"much"
Topic 2: 0.081*"biryani" + 0.057*"chicken" + 0.041*"good" + 0.035*"really" + 0.027*"meghana" + 0.023*"special" + 0.021*"food" + 0.021*"tried"
Topic 3: 0.038*"amazing" + 0.036*"worth" + 0.030*"service" + 0.029*"food" + 0.027*"dosa" + 0.025*"place" + 0.024*"delicious" + 0.023*"meals"
Topic 4: 0.081*"dosa" + 0.051*"masala" + 0.041*"best" + 0.038*"bangalore" + 0.033*"one" + 0.027*"crispy" + 0.022*"vidyarthi" + 0.022*"bhavan"
Topic 5: 0.089*"place" + 0.029*"dosa" + 0.025*"quite" + 0.024*"visited" + 0.020*"hyped" + 0.017*"koramangala" + 0.017*"well" + 0.016*"average"
Topic 6: 0.046*"food" + 0.044*"best" + 0.036*"indian" + 0.034*"bangalore" + 0.033*"south" + 0.028*"one" + 0.028*"place" + 0.025*"breakfast"


### Manually label each topic after reading the top words

In [6]:
topic_labels = {
    i: f"TODO: label topic {i}" for i in range(num_topics)
}
topic_labels

{0: 'TODO: label topic 0',
 1: 'TODO: label topic 1',
 2: 'TODO: label topic 2',
 3: 'TODO: label topic 3',
 4: 'TODO: label topic 4',
 5: 'TODO: label topic 5',
 6: 'TODO: label topic 6',
 7: 'TODO: label topic 7'}

### Assign the dominant topic to each review

In [7]:
def get_dominant_topic(bow):
    topics = lda_model.get_document_topics(bow)
    return max(topics, key=lambda x: x[1])[0] if topics else None
 
df["dominant_topic"] = [get_dominant_topic(bow) for bow in corpus]
df["dominant_topic"].value_counts()

dominant_topic
7    168
2     88
0     86
4     60
6     53
5     49
1     42
3     41
Name: count, dtype: int64

### Save the model

In [8]:
lda_model.save("../models/lda_model.gensim")
dictionary.save("../models/lda_dictionary.gensim")
print("Saved.")

Saved.
